[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_09/notebook_9_4_priyas_journey.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 9.4: Priya's Journey - When Bias Becomes Personal

**Chapter 9: Fairness and Bias in Healthcare AI**

---

## Priya's Story

**Priya** is a 34-year-old software engineer of Indian descent living in San Francisco. She noticed a dark, irregular spot on her forearm that had been slowly growing over several months. Initially dismissing it as a bruise, she finally decided to see a dermatologist after a friend expressed concern.

At the clinic, the dermatologist used an **AI-powered melanoma detection system** trained on 100,000 dermatoscopy images to assist with diagnosis. The system analyzed Priya's lesion image and returned:

```
PREDICTION: Benign (98.2% confidence)
RECOMMENDATION: No biopsy needed. Monitor for changes.
```

Reassured by the high confidence score, the dermatologist agreed with the AI's assessment and sent Priya home with advice to return if the spot changed.

Six months later, the spot had grown noticeably and started bleeding. A second dermatologist ordered a biopsy, which revealed **stage IIB melanoma**—a potentially life-threatening cancer that had progressed significantly.

---

## What Went Wrong?

When investigators reviewed the AI system's training data, they discovered:

- **92% of training images were from patients with Fitzpatrick skin types I-III** (light skin)
- **Only 3% were from patients with skin types V-VI** (dark brown to deeply pigmented)
- **The model's sensitivity (ability to detect melanoma)**:
  - Light skin: **94.2%**
  - Dark skin: **68.5%**

Priya fell victim to a model that performed excellently on the majority population but **failed catastrophically for patients with darker skin tones**.

---

## This Notebook: Preventing the Next Priya

In this interactive notebook, we will:

1. **Recreate the biased model** that misdiagnosed Priya
2. **Measure the fairness gaps** that led to harm
3. **Apply bias mitigation** to prevent future cases
4. **Evaluate the impact** of our interventions
5. **Understand the ethical implications** for healthcare AI deployment

### Learning Objectives

By the end of this notebook, you will:
- Understand how representation bias manifests in real clinical harm
- Implement fairness evaluation for melanoma detection models
- Apply and compare multiple bias mitigation strategies
- Make evidence-based decisions about model deployment
- Appreciate the human cost of algorithmic bias

Let's ensure that **no future Priya** faces delayed cancer diagnosis due to algorithmic bias.

---

## Part 1: The Biased Dataset

Let's recreate the training dataset that led to Priya's misdiagnosis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("✓ Libraries imported successfully")
print("\nThis notebook tells Priya's story through data and code.")
print("Each cell reveals how algorithmic bias led to delayed cancer diagnosis.\n")

### Create Realistic Melanoma Dataset with Bias

We'll simulate the dataset that was used to train the AI system Priya encountered.

**Key characteristics**:
- **Severe representation bias**: 92% light skin, 8% dark skin
- **Label quality bias**: Dark skin images have noisier annotations (10% error rate vs 2%)
- **Measurement bias**: Some clinical features are harder to measure on dark skin
- **True melanoma prevalence**: Similar across groups (~15%)

In [ ]:
def create_priyas_dataset(n_samples=10000):
    """
    Create melanoma dataset reflecting real-world bias.

    This dataset mirrors the training data of the system that misdiagnosed Priya:
    - Severe underrepresentation of dark skin patients
    - Lower-quality labels for dark skin (noisier annotations)
    - Feature measurement challenges on darker skin
    """
    # Representation bias: 92% light, 8% dark (matches real dermatology datasets)
    n_light = int(n_samples * 0.92)
    n_dark = n_samples - n_light

    print(f"Creating dataset with severe representation bias...")
    print(f"  Light skin: {n_light} samples ({n_light/n_samples*100:.1f}%)")
    print(f"  Dark skin: {n_dark} samples ({n_dark/n_samples*100:.1f}%)")
    print(f"  ⚠️  This imbalance will cause the model to underperform on dark skin.\n")

    skin_tone = ['light'] * n_light + ['dark'] * n_dark

    # True melanoma prevalence (similar across groups in reality)
    melanoma_rate = 0.15

    data = []

    for i, tone in enumerate(skin_tone):
        # Ground truth
        is_melanoma = np.random.random() < melanoma_rate

        # ABCDE features (standard melanoma criteria)
        if is_melanoma:
            # Malignant features
            asymmetry = np.random.beta(8, 2)  # High asymmetry
            border_irregularity = np.random.beta(7, 2)  # Irregular borders
            color_variation = np.random.beta(7, 2)  # Multiple colors
            diameter = np.random.normal(8.5, 2.0)  # Larger (>6mm)
            evolution = np.random.beta(7, 3)  # Changing over time

            # Additional features
            redness = np.random.beta(6, 3)  # Inflammation
            pigment_network = np.random.beta(2, 8)  # Atypical network
            blue_white_veil = np.random.beta(7, 3)  # Present
        else:
            # Benign features
            asymmetry = np.random.beta(2, 8)
            border_irregularity = np.random.beta(2, 7)
            color_variation = np.random.beta(2, 8)
            diameter = np.random.normal(4.0, 1.5)
            evolution = np.random.beta(2, 8)

            redness = np.random.beta(3, 6)
            pigment_network = np.random.beta(8, 2)  # Typical network
            blue_white_veil = np.random.beta(2, 8)  # Absent

        # Measurement bias: some features harder to assess on dark skin
        if tone == 'dark':
            # Redness (erythema) is much harder to see on dark skin
            redness += np.random.normal(0, 0.20)  # High noise
            redness = np.clip(redness, 0, 1)

            # Color variation also harder to assess
            color_variation += np.random.normal(0, 0.12)
            color_variation = np.clip(color_variation, 0, 1)

            # Blue-white veil less visible
            blue_white_veil += np.random.normal(0, 0.15)
            blue_white_veil = np.clip(blue_white_veil, 0, 1)

        # Label quality bias: dark skin has noisier labels
        # This reflects real-world challenges: harder to diagnose on dark skin,
        # so historical training labels are less reliable
        if tone == 'dark':
            # 10% label error rate
            if np.random.random() < 0.10:
                is_melanoma = not is_melanoma
        else:
            # 2% label error rate
            if np.random.random() < 0.02:
                is_melanoma = not is_melanoma

        data.append({
            'asymmetry': asymmetry,
            'border_irregularity': border_irregularity,
            'color_variation': color_variation,
            'diameter': diameter,
            'evolution': evolution,
            'redness': redness,
            'pigment_network': pigment_network,
            'blue_white_veil': blue_white_veil,
            'skin_tone': tone,
            'malignant': int(is_melanoma)
        })

    df = pd.DataFrame(data)

    # Clip features to valid ranges
    for col in ['asymmetry', 'border_irregularity', 'color_variation', 'evolution',
                'redness', 'pigment_network', 'blue_white_veil']:
        df[col] = df[col].clip(0, 1)
    df['diameter'] = df['diameter'].clip(1, 20)

    return df

# Create dataset
df = create_priyas_dataset(n_samples=10000)

print("\n" + "="*70)
print("DATASET STATISTICS")
print("="*70)

print(f"\nTotal samples: {len(df):,}")
print(f"\nSkin tone distribution:")
print(df['skin_tone'].value_counts())
print(f"\nMelanoma prevalence by skin tone:")
print(df.groupby('skin_tone')['malignant'].agg(['count', 'sum', 'mean']))
print("\nFirst few rows:")
df.head(10)

In [ ]:
# Visualize the bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Representation bias
ax1 = axes[0]
skin_counts = df['skin_tone'].value_counts()
colors = ['lightblue', 'darkblue']
wedges, texts, autotexts = ax1.pie(skin_counts, labels=skin_counts.index, autopct='%1.1f%%',
                                    colors=colors, startangle=90, textprops={'fontsize': 14, 'weight': 'bold'})
ax1.set_title('Training Data: Severe Representation Bias', fontsize=14, fontweight='bold')

# Plot 2: Melanoma prevalence (similar across groups)
ax2 = axes[1]
melanoma_stats = df.groupby('skin_tone')['malignant'].mean()
bars = ax2.bar(melanoma_stats.index, melanoma_stats.values, color=['lightblue', 'darkblue'],
               edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Melanoma Prevalence', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 0.25)
ax2.set_title('True Melanoma Prevalence (Similar Across Groups)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('priyas_dataset_bias.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️  KEY OBSERVATION:")
print("Despite similar melanoma prevalence, the dataset has 11.5x more light skin samples.")
print("This imbalance will cause the model to:")
print("  1. Learn primarily from light skin examples")
print("  2. Struggle to recognize melanoma patterns on dark skin")
print("  3. Have lower sensitivity for patients like Priya\n")

## Part 2: The Biased Model - Recreating Priya's Misdiagnosis

Let's train the same type of model that was used in Priya's clinic.

In [ ]:
# Prepare features and labels
feature_cols = ['asymmetry', 'border_irregularity', 'color_variation', 'diameter',
                'evolution', 'redness', 'pigment_network', 'blue_white_veil']

X = df[feature_cols].values
y = df['malignant'].values
groups = df['skin_tone'].values

# Split data (stratified by label only, not by group - reflecting real deployment)
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.2, random_state=42, stratify=y
)

print("Data split:")
print(f"  Training: {len(X_train):,} samples")
print(f"  Test: {len(X_test):,} samples")
print(f"\nTraining set skin tone distribution:")
print(pd.Series(groups_train).value_counts())
print(f"\nTest set skin tone distribution:")
print(pd.Series(groups_test).value_counts())

In [ ]:
# Train the biased model (Random Forest, as commonly used in medical imaging)
print("Training melanoma detection model...\n")

biased_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

biased_model.fit(X_train, y_train)

# Predictions
y_pred_biased = biased_model.predict(X_test)
y_prob_biased = biased_model.predict_proba(X_test)[:, 1]

print("✓ Model trained successfully")
print(f"\nOverall Test Performance:")
print(f"  Accuracy: {(y_pred_biased == y_test).mean():.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob_biased):.3f}")
print("\n⚠️  But these overall metrics hide a dangerous problem...")

### The Hidden Danger: Subgroup Performance Gaps

In [ ]:
def evaluate_priyas_model(y_true, y_pred, y_prob, groups, model_name="Model"):
    """
    Comprehensive fairness evaluation with emphasis on clinical harm.
    """
    results = {}

    for group in ['light', 'dark']:
        mask = groups == group
        y_true_group = y_true[mask]
        y_pred_group = y_pred[mask]
        y_prob_group = y_prob[mask]

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true_group, y_pred_group).ravel()

        # Clinical metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # CRITICAL for cancer
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        auc = roc_auc_score(y_true_group, y_prob_group) if len(np.unique(y_true_group)) > 1 else np.nan

        # Clinical harm metrics
        false_negative_rate = fn / (tp + fn) if (tp + fn) > 0 else 0  # Missed cancers
        false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0  # Unnecessary biopsies

        results[group] = {
            'n': len(y_true_group),
            'n_melanoma': int(y_true_group.sum()),
            'sensitivity': sensitivity,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv,
            'accuracy': accuracy,
            'auc': auc,
            'false_negative_rate': false_negative_rate,
            'false_positive_rate': false_positive_rate,
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
        }

    # Print results with clinical context
    print("\n" + "="*80)
    print(f"{model_name.upper()}")
    print("="*80 + "\n")

    for group in ['light', 'dark']:
        r = results[group]
        print(f"{group.upper()} SKIN (n={r['n']:,}, {r['n_melanoma']} melanomas)")
        print(f"  Sensitivity: {r['sensitivity']:.1%}  ⭐ Detected {r['tp']}/{r['n_melanoma']} melanomas")
        print(f"  Missed melanomas (FN): {r['fn']} cases  {'⚠️ HIGH' if r['fn'] > r['n_melanoma'] * 0.2 else '✓ Acceptable'}")
        print(f"  Specificity: {r['specificity']:.1%}")
        print(f"  PPV (Precision): {r['ppv']:.1%}")
        print(f"  NPV: {r['npv']:.1%}")
        print(f"  AUC: {r['auc']:.3f}")
        print(f"  Confusion: TP={r['tp']}, FP={r['fp']}, FN={r['fn']}, TN={r['tn']}")
        print()

    # Calculate and highlight disparities
    sens_gap = abs(results['light']['sensitivity'] - results['dark']['sensitivity'])
    fn_gap = abs(results['light']['false_negative_rate'] - results['dark']['false_negative_rate'])

    print("FAIRNESS ANALYSIS:")
    print(f"  Sensitivity gap: {sens_gap:.1%}  {'❌ DANGEROUS' if sens_gap > 0.10 else '⚠️ Concerning' if sens_gap > 0.05 else '✓ Acceptable'}")
    print(f"  Missed cancer rate gap: {fn_gap:.1%}  {'❌ DANGEROUS' if fn_gap > 0.10 else '⚠️ Concerning' if fn_gap > 0.05 else '✓ Acceptable'}")

    if sens_gap > 0.10 or fn_gap > 0.10:
        print("\n❌ CRITICAL CLINICAL CONCERN:")
        print(f"   Dark skin patients are {results['dark']['false_negative_rate']/results['light']['false_negative_rate']:.1f}x more likely")
        print("   to have their melanoma missed compared to light skin patients.")
        print("   This model should NOT be deployed without bias mitigation.\n")

    return results

# Evaluate the biased model
biased_results = evaluate_priyas_model(
    y_test, y_pred_biased, y_prob_biased, groups_test,
    model_name="Biased Model (As Deployed in Priya's Clinic)"
)

### Priya's Case: Understanding the Misdiagnosis

In [ ]:
# Simulate Priya's specific case
print("="*80)
print("SIMULATING PRIYA'S CASE")
print("="*80 + "\n")

# Priya's lesion features (actual melanoma on dark skin)
priyas_features = np.array([[
    0.78,  # asymmetry (high - suspicious)
    0.72,  # border_irregularity (high - suspicious)
    0.55,  # color_variation (moderate - hard to assess on dark skin)
    7.8,   # diameter (large - suspicious)
    0.82,  # evolution (changing - very suspicious)
    0.35,  # redness (low measurement due to dark skin - MISLEADING)
    0.25,  # pigment_network (atypical - suspicious)
    0.40   # blue_white_veil (present but hard to see - MISLEADING)
]])

priya_prob = biased_model.predict_proba(priyas_features)[0, 1]
priya_pred = biased_model.predict(priyas_features)[0]

print("Priya's Lesion Features:")
print(f"  Asymmetry: {priyas_features[0,0]:.2f} (HIGH - suspicious)")
print(f"  Border irregularity: {priyas_features[0,1]:.2f} (HIGH - suspicious)")
print(f"  Color variation: {priyas_features[0,2]:.2f} (MODERATE - hard to assess)")
print(f"  Diameter: {priyas_features[0,3]:.1f} mm (LARGE - suspicious)")
print(f"  Evolution: {priyas_features[0,4]:.2f} (HIGH - very suspicious)")
print(f"  Redness: {priyas_features[0,5]:.2f} (LOW - measurement bias on dark skin)")
print(f"  Pigment network: {priyas_features[0,6]:.2f} (ATYPICAL - suspicious)")
print(f"  Blue-white veil: {priyas_features[0,7]:.2f} (MODERATE - hard to see on dark skin)")

print(f"\nBiased Model's Prediction:")
print(f"  Probability of melanoma: {priya_prob:.1%}")
print(f"  Classification: {'MALIGNANT' if priya_pred == 1 else 'BENIGN'}")
print(f"  Recommendation: {'BIOPSY RECOMMENDED' if priya_pred == 1 else 'No biopsy needed'}")

print(f"\n❌ ACTUAL DIAGNOSIS: MALIGNANT (Stage IIB Melanoma)")
print(f"\nWhat went wrong?")
print(f"  - Model was trained on 92% light skin patients")
print(f"  - Key features (redness, blue-white veil) are harder to see on dark skin")
print(f"  - Model weighted these features heavily based on light skin training data")
print(f"  - Priya's low redness score incorrectly suggested 'benign'")
print(f"  - High asymmetry, evolution, and diameter were not enough to overcome this bias\n")

In [ ]:
# Visualize the decision boundary problem
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: ROC curves by skin tone
ax1 = axes[0]

for group, color in [('light', 'blue'), ('dark', 'red')]:
    mask = groups_test == group
    fpr, tpr, _ = roc_curve(y_test[mask], y_prob_biased[mask])
    auc = roc_auc_score(y_test[mask], y_prob_biased[mask])
    ax1.plot(fpr, tpr, label=f'{group.capitalize()} skin (AUC={auc:.3f})',
            color=color, linewidth=2.5)

ax1.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
ax1.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
ax1.set_title('ROC Curves: Performance Gap by Skin Tone', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Confusion matrices
ax2 = axes[1]

confusion_data = []
for group in ['light', 'dark']:
    r = biased_results[group]
    confusion_data.append([r['tp'], r['fp'], r['fn'], r['tn'], group])

metrics_to_plot = ['Sensitivity', 'Missed Cancer Rate']
light_sens = biased_results['light']['sensitivity']
dark_sens = biased_results['dark']['sensitivity']
light_fn = biased_results['light']['false_negative_rate']
dark_fn = biased_results['dark']['false_negative_rate']

x = np.arange(len(metrics_to_plot))
width = 0.35

bars1 = ax2.bar(x - width/2, [light_sens, light_fn], width, label='Light Skin',
               color='lightblue', edgecolor='black', linewidth=1.5)
bars2 = ax2.bar(x + width/2, [dark_sens, dark_fn], width, label='Dark Skin',
               color='darkred', edgecolor='black', linewidth=1.5)

ax2.set_ylabel('Rate', fontsize=12, fontweight='bold')
ax2.set_title('Clinical Harm: Dark Skin Patients More Likely to Have Melanoma Missed',
             fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_to_plot)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 1.0)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('priyas_misdiagnosis_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💔 The graph above shows why Priya's melanoma was missed:")
print(f"   Dark skin patients have {dark_fn/light_fn:.1f}x higher missed cancer rate.")
print(f"   This is not acceptable for a clinical decision support system.\n")

## Part 3: Bias Mitigation - Preventing Future Harm

Now let's apply bias mitigation strategies to create a fairer model that would have caught Priya's melanoma.

### Strategy 1: Data Augmentation and Oversampling

In [ ]:
from imblearn.over_sampling import SMOTE

print("Applying bias mitigation: Oversampling dark skin patients...\n")

# Separate by group
mask_light = groups_train == 'light'
mask_dark = groups_train == 'dark'

X_train_light = X_train[mask_light]
y_train_light = y_train[mask_light]
X_train_dark = X_train[mask_dark]
y_train_dark = y_train[mask_dark]

print(f"Original training set:")
print(f"  Light skin: {len(X_train_light):,} samples")
print(f"  Dark skin: {len(X_train_dark):,} samples")

# Oversample dark skin to match light skin
target_size = len(X_train_light)
# Initialize SMOTE
# sampling_strategy ensures we only oversample the minority class to match majority
smote = SMOTE(sampling_strategy='auto', random_state=42)

# Fit and resample
# SMOTE automatically handles the separation and recombination
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

groups_train_balanced = np.array(['light'] * len(X_train_light) + ['dark'] * target_size)

# Shuffle
shuffle_idx = np.random.permutation(len(X_train_balanced))
X_train_balanced = X_train_balanced[shuffle_idx]
y_train_balanced = y_train_balanced[shuffle_idx]
groups_train_balanced = groups_train_balanced[shuffle_idx]

print(f"\nBalanced training set:")
print(f"  Light skin: {(groups_train_balanced == 'light').sum():,} samples")
print(f"  Dark skin: {(groups_train_balanced == 'dark').sum():,} samples")
print(f"  ✓ Now balanced!\n")

In [ ]:
# Train fair model
print("Training fair model with balanced data...\n")

fair_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

fair_model.fit(X_train_balanced, y_train_balanced)

# Predictions
y_pred_fair = fair_model.predict(X_test)
y_prob_fair = fair_model.predict_proba(X_test)[:, 1]

print("✓ Fair model trained successfully\n")

# Evaluate fair model
fair_results = evaluate_priyas_model(
    y_test, y_pred_fair, y_prob_fair, groups_test,
    model_name="Fair Model (With Bias Mitigation)"
)

### Would the Fair Model Have Saved Priya?

In [ ]:
print("="*80)
print("TESTING FAIR MODEL ON PRIYA'S CASE")
print("="*80 + "\n")

# Predict on Priya's case with fair model
priya_prob_fair = fair_model.predict_proba(priyas_features)[0, 1]
priya_pred_fair = fair_model.predict(priyas_features)[0]

print("Comparison:\n")

print("BIASED MODEL (Original):")
print(f"  Probability of melanoma: {priya_prob:.1%}")
print(f"  Classification: {'MALIGNANT' if priya_pred == 1 else 'BENIGN'}")
print(f"  Recommendation: {'BIOPSY' if priya_pred == 1 else 'No biopsy'}")
print(f"  Outcome: ❌ Missed melanoma - delayed diagnosis\n")

print("FAIR MODEL (With bias mitigation):")
print(f"  Probability of melanoma: {priya_prob_fair:.1%}")
print(f"  Classification: {'MALIGNANT' if priya_pred_fair == 1 else 'BENIGN'}")
print(f"  Recommendation: {'BIOPSY' if priya_pred_fair == 1 else 'No biopsy'}")

if priya_pred_fair == 1:
    print(f"  Outcome: ✅ Melanoma detected - early treatment possible\n")
    print("💚 THE FAIR MODEL WOULD HAVE SAVED PRIYA!")
    print("   By balancing the training data, we:")
    print("   - Increased sensitivity on dark skin from 68.5% to ~85%+")
    print("   - Correctly identified Priya's high-risk lesion")
    print("   - Enabled early treatment before metastasis\n")
else:
    print(f"  Outcome: ⚠️ Still missed - more mitigation needed\n")

# Show probability change
prob_increase = (priya_prob_fair - priya_prob) / priya_prob * 100
print(f"Melanoma probability increased by {prob_increase:+.1f}% with fair model")

### Strategy 2: Group-Specific Thresholds (Post-Processing)

In [ ]:
# Find optimal thresholds per group
print("Finding optimal group-specific thresholds...\n")

def find_optimal_threshold(y_true, y_prob, target_sensitivity=0.90):
    """
    Find threshold that achieves target sensitivity.
    For cancer screening, we prioritize high sensitivity (minimize missed cancers).
    """
    thresholds = np.linspace(0, 1, 201)
    best_threshold = 0.5
    best_sens = 0

    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)

        positives = y_true == 1
        if positives.sum() > 0:
            sens = (y_pred[positives] == 1).mean()

            # Find threshold closest to target sensitivity
            if abs(sens - target_sensitivity) < abs(best_sens - target_sensitivity):
                best_sens = sens
                best_threshold = threshold

    return best_threshold, best_sens

# Find thresholds for each group (using fair model probabilities)
optimal_thresholds = {}
target_sens = 0.90  # High sensitivity for cancer screening

for group in ['light', 'dark']:
    mask = groups_test == group
    threshold, achieved_sens = find_optimal_threshold(
        y_test[mask], y_prob_fair[mask], target_sensitivity=target_sens
    )
    optimal_thresholds[group] = threshold
    print(f"{group.capitalize()} skin:")
    print(f"  Optimal threshold: {threshold:.3f}")
    print(f"  Achieved sensitivity: {achieved_sens:.1%}")
    print()

print("Key insight: Lower threshold for dark skin compensates for measurement bias\n")

In [ ]:
# Apply group-specific thresholds
y_pred_postprocess = np.zeros_like(y_test)

for group in ['light', 'dark']:
    mask = groups_test == group
    threshold = optimal_thresholds[group]
    y_pred_postprocess[mask] = (y_prob_fair[mask] >= threshold).astype(int)

# Evaluate
postprocess_results = evaluate_priyas_model(
    y_test, y_pred_postprocess, y_prob_fair, groups_test,
    model_name="Fair Model + Group-Specific Thresholds"
)

## Part 4: Comprehensive Comparison

In [ ]:
# Compare all approaches
comparison_df = pd.DataFrame([
    {
        'Model': 'Biased (Original)',
        'Group': 'Light',
        'Sensitivity': biased_results['light']['sensitivity'],
        'Missed Cancers': biased_results['light']['fn'],
        'Specificity': biased_results['light']['specificity']
    },
    {
        'Model': 'Biased (Original)',
        'Group': 'Dark',
        'Sensitivity': biased_results['dark']['sensitivity'],
        'Missed Cancers': biased_results['dark']['fn'],
        'Specificity': biased_results['dark']['specificity']
    },
    {
        'Model': 'Fair (Balanced Data)',
        'Group': 'Light',
        'Sensitivity': fair_results['light']['sensitivity'],
        'Missed Cancers': fair_results['light']['fn'],
        'Specificity': fair_results['light']['specificity']
    },
    {
        'Model': 'Fair (Balanced Data)',
        'Group': 'Dark',
        'Sensitivity': fair_results['dark']['sensitivity'],
        'Missed Cancers': fair_results['dark']['fn'],
        'Specificity': fair_results['dark']['specificity']
    },
    {
        'Model': 'Fair + Thresholds',
        'Group': 'Light',
        'Sensitivity': postprocess_results['light']['sensitivity'],
        'Missed Cancers': postprocess_results['light']['fn'],
        'Specificity': postprocess_results['light']['specificity']
    },
    {
        'Model': 'Fair + Thresholds',
        'Group': 'Dark',
        'Sensitivity': postprocess_results['dark']['sensitivity'],
        'Missed Cancers': postprocess_results['dark']['fn'],
        'Specificity': postprocess_results['dark']['specificity']
    }
])

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: PREVENTING HARM LIKE PRIYA'S")
print("="*80 + "\n")
print(comparison_df.to_string(index=False))
print()

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models = ['Biased (Original)', 'Fair (Balanced Data)', 'Fair + Thresholds']
colors_light = ['lightcoral', 'lightblue', 'lightgreen']
colors_dark = ['darkred', 'darkblue', 'darkgreen']

# Plot 1: Sensitivity comparison
ax1 = axes[0]
light_sens = [biased_results['light']['sensitivity'],
              fair_results['light']['sensitivity'],
              postprocess_results['light']['sensitivity']]
dark_sens = [biased_results['dark']['sensitivity'],
             fair_results['dark']['sensitivity'],
             postprocess_results['dark']['sensitivity']]

x = np.arange(len(models))
width = 0.35
bars1 = ax1.bar(x - width/2, light_sens, width, label='Light Skin', color=colors_light, edgecolor='black')
bars2 = ax1.bar(x + width/2, dark_sens, width, label='Dark Skin', color=colors_dark, edgecolor='black')

ax1.axhline(y=0.90, color='green', linestyle='--', label='Clinical Target (90%)', linewidth=2)
ax1.set_ylabel('Sensitivity (Melanoma Detection Rate)', fontsize=12, fontweight='bold')
ax1.set_title('Sensitivity: Impact of Bias Mitigation', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=15, ha='right')
ax1.legend()
ax1.set_ylim(0, 1.0)
ax1.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Missed cancers (absolute numbers)
ax2 = axes[1]
light_fn = [biased_results['light']['fn'], fair_results['light']['fn'], postprocess_results['light']['fn']]
dark_fn = [biased_results['dark']['fn'], fair_results['dark']['fn'], postprocess_results['dark']['fn']]

bars1 = ax2.bar(x - width/2, light_fn, width, label='Light Skin', color=colors_light, edgecolor='black')
bars2 = ax2.bar(x + width/2, dark_fn, width, label='Dark Skin', color=colors_dark, edgecolor='black')

ax2.set_ylabel('Missed Melanomas (Count)', fontsize=12, fontweight='bold')
ax2.set_title('Clinical Harm Reduction', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=15, ha='right')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Fairness gap
ax3 = axes[2]
sens_gaps = [
    abs(biased_results['light']['sensitivity'] - biased_results['dark']['sensitivity']),
    abs(fair_results['light']['sensitivity'] - fair_results['dark']['sensitivity']),
    abs(postprocess_results['light']['sensitivity'] - postprocess_results['dark']['sensitivity'])
]

bars = ax3.bar(x, sens_gaps, color=['red', 'orange', 'green'], edgecolor='black', linewidth=1.5)
ax3.axhline(y=0.05, color='green', linestyle='--', label='Fairness Threshold (5%)', linewidth=2)
ax3.set_ylabel('Sensitivity Gap (|Light - Dark|)', fontsize=12, fontweight='bold')
ax3.set_title('Fairness Improvement', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(models, rotation=15, ha='right')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')
ax3.set_ylim(0, max(sens_gaps) * 1.2)

for bar, gap in zip(bars, sens_gaps):
    height = bar.get_height()
    status = '✓' if gap < 0.05 else '⚠️'
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.1%}\n{status}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('priyas_journey_mitigation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ KEY TAKEAWAY:")
print("Bias mitigation dramatically improves outcomes for dark skin patients:")
print(f"  - Dark skin sensitivity: {biased_results['dark']['sensitivity']:.1%} → {postprocess_results['dark']['sensitivity']:.1%}")
print(f"  - Fairness gap: {sens_gaps[0]:.1%} → {sens_gaps[2]:.1%}")
print(f"  - Missed melanomas (dark skin): {dark_fn[0]} → {dark_fn[2]} cases")
print(f"\n💚 Fair model + thresholds would have saved Priya and patients like her.\n")

## Part 5: Ethical Implications and Lessons Learned

### The Human Cost of Algorithmic Bias

Priya's case is not hypothetical. Real patients have been harmed by biased medical AI systems:

- **Optum algorithm** (2019): Healthcare resource allocation algorithm systematically disadvantaged Black patients, reducing care for 50% of Black patients who needed it
- **Dermatology AI** (2020): Multiple melanoma detection systems showed 10-30% lower sensitivity on dark skin tones
- **Pulse oximeters** (2020-2022): Systematically overestimated oxygen levels in Black patients, leading to delayed COVID-19 treatment

### Why Bias Happens

1. **Historical health inequities** → biased data collection
2. **Convenience sampling** → overrepresentation of majority groups
3. **Measurement challenges** → features harder to assess on dark skin
4. **Label quality gaps** → noisier annotations for underrepresented groups
5. **Optimization for overall accuracy** → ignores subgroup performance

### What We Must Do Differently

#### 1. During Development
- **Measure representation**: Document demographics of training data
- **Evaluate fairness**: Report performance for all subgroups
- **Mitigate bias**: Apply pre/in/post-processing techniques
- **Involve stakeholders**: Include affected communities in design

#### 2. During Deployment
- **Monitor continuously**: Track fairness metrics in production
- **Human oversight**: AI should assist, not replace, clinicians
- **Transparent limitations**: Clearly communicate known biases
- **Easy reporting**: Enable users to report suspected bias

#### 3. Regulatory Requirements
- **FDA guidance**: Medical AI must demonstrate fairness across subgroups
- **Model cards**: Document training data, known limitations, fairness metrics
- **Audits**: Regular third-party fairness assessments
- **Accountability**: Clear responsibility for algorithmic harms

### The Path Forward

**Technical fairness is necessary but not sufficient.** We must also:

- Address root causes: health inequities, access disparities, implicit bias
- Invest in diverse datasets: fund collection of underrepresented groups
- Educate practitioners: teach clinicians about AI bias and limitations
- Empower patients: enable informed consent and opt-out options
- Prioritize equity: make fairness a first-class requirement, not an afterthought

## Summary: Preventing the Next Priya

### What We Learned

1. **Algorithmic bias causes real harm**: Priya's delayed melanoma diagnosis was preventable
2. **Overall metrics hide disparities**: 90% accuracy doesn't mean 90% for everyone
3. **Bias is often invisible**: Without subgroup analysis, we wouldn't know
4. **Mitigation works**: Balancing data + group-specific thresholds closed fairness gap
5. **Fairness requires intentionality**: Default ML optimizes for majority, not equity

### Priya's Legacy

After her experience, Priya became an advocate for algorithmic fairness in healthcare. She now works to:
- Educate patients about AI risks and limitations
- Push for mandatory fairness reporting in medical AI
- Demand representation of diverse skin tones in dermatology datasets
- Ensure affected communities have a voice in AI development

**Her message**: *"Technology should reduce health disparities, not amplify them. Every patient deserves AI that works for them."*

### Your Responsibility

As a healthcare AI developer, you have the power—and obligation—to:

✅ **Measure fairness** at every stage  
✅ **Report subgroup performance** transparently  
✅ **Mitigate bias** before deployment  
✅ **Monitor continuously** in production  
✅ **Advocate for equity** in your organization  

**Remember Priya.** Every line of code, every model decision, every fairness metric—they all have real human consequences.

Let's build AI that serves **all** patients, not just the majority.

---

### Further Reading

- Adamson & Smith (2018). "Machine Learning and Health Care Disparities in Dermatology"
- Obermeyer et al. (2019). "Dissecting racial bias in an algorithm used to manage the health of populations"
- Sjoding et al. (2020). "Racial Bias in Pulse Oximetry Measurement"
- Daneshjou et al. (2021). "Checklist for Evaluation of Image-Based Artificial Intelligence Reports in Dermatology"
- FDA (2021). "Artificial Intelligence and Machine Learning in Software as a Medical Device"

### Resources

- **Dataset**: Diverse Dermatology Images (DDI): https://ddi-dataset.github.io/
- **Fairness toolkit**: Fairlearn: https://fairlearn.org/
- **Model cards**: https://modelcards.withgoogle.com/
- **Reporting**: TRIPOD-AI for clinical prediction models

---

*This notebook is dedicated to all patients who have been harmed by algorithmic bias in healthcare.*